# Lab 3 — Debugging a Sentiment Pipeline with DSPY

<a href="https://colab.research.google.com/github/Tulane-CMPS-1010-AI-Systems/course-materials/blob/main/labs/03-pipeline_lab.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

This lab introduces **LLM pipelines** using `dspy`. You will:
- Build a simple **sentiment analysis pipeline**.
- Compare a **modular pipeline** to a **single prompt**.
- Apply the **scientific method** to reason about debugging and modularity.
- Deploy your pipeline interactively using **Gradio**.



## 🎯 Learning Goals
By the end of this lab, you will be able to:
- Build a multi-step **LLM pipeline** in DSPY.
- Handle errors gracefully using `safe_run`.
- Compare **single-prompt vs. pipeline** approaches.
- Log and interpret intermediate results.
- Launch a **Gradio app** to visualize and test your system.

## 📚 Lecture Connection

**Before starting this lab, make sure you understand:**
- From **Day 3 Lecture**: What is a pipeline?
- From **Day 3 Lecture**: How does `dspy.Predict()` work?
- From **Day 3 Lecture**: What does `dspy.inspect()` show?

**This lab applies:**
- Building modular pipelines (Lecture Section 3)
- Using DSPy to structure LLM calls (Lecture Section 4)
- Debugging by decomposition (Lecture Section 5)

If you need a refresher, review Lecture Day 3.

---

## ✅ Before You Start Checklist

Before beginning this lab, make sure you can:
- [ ] Call a dspy.Predict() function (review Lecture Cell 10 if needed)
- [ ] Understand what a Python function returns
- [ ] Use a for loop in Python
- [ ] Understand try/except error handling (basic)

**If you're unsure about any of these, review the Day 3 lecture first!**

---


In [ ]:

# @title 🔧 Lab 3 Setup (Run this first)
!git clone --depth 1 -q https://github.com/Tulane-CMPS-1010-AI-Systems/course-materials.git
import sys; sys.path.append('./course-materials')
from course_utils import lab3_setup
lab3_setup()

## 💬 Pre-Lab Questions

> ✏️ *Edit this cell and write your answers below each question.*

Answer each in 1-2 sentences:

1. Why might we prefer breaking a long, complex prompt into smaller pipeline steps?
2. What types of errors do you think could happen in a multi-step system?
3. What might make debugging easier when using multiple smaller modules?

## 🧠 Scientific Question & Hypothesis
**Question:** How does modular design make LLM pipelines easier to debug than single prompts?

**Hypothesis:** I expect that the pipeline version will be easier to understand and fix when errors occur, because it exposes each intermediate step.

In [ ]:
import dspy
import os

lm = dspy.LM("openai/gpt-4o-mini") # , api_key=os.environ["OPENAI_API_KEY"])
dspy.configure(lm=lm)
print("✅ DSPy configured. Ready to build pipelines!")

# Part 1 – Build Your Own Sentiment Pipeline

We'll build this step-by-step. Don't worry if it seems like a lot—we'll do it one piece at a time!

**Our goal:** Build a pipeline that:
1. Extracts sentences about Tulane University from text
2. Annotates each sentence with sentiment (positive/negative/neutral)
3. Summarizes the overall sentiment

---

## Part 1a – Step 1: Extract Sentences

Let's start with just the extraction step. This will help you understand how `dspy.Predict()` works.

In [ ]:
# @title Test the Extract Step (Try this first!)
import dspy

# Create the extract predictor
extract = dspy.Predict('text -> sentences_about_tulane: list[str]')

# Test it on a simple example
test_text = "Tulane University is great. Other schools are fine too."
result = extract(text=test_text)

# Print what we got
print("Result type:", type(result))
print("Result:", result)
print("\n✅ Sentences extracted:", result.sentences_about_tulane)

# 💡 Hint: Notice that result has a .sentences_about_tulane attribute
# This matches the output field name in the Predict() signature
#
# ⚠️ Note: If you see any Pydantic warnings, don't worry! They're harmless.
# The code is working correctly - you can see the extracted sentences above.


## Part 1b – Step 2: Annotate Sentiment

Now let's test the annotation step. This will label each sentence as positive, negative, or neutral.


In [ ]:
# @title Test the Annotate Step
# Create the annotate predictor
annotate = dspy.Predict('sentence -> sentiment: str')

# Test it on a single sentence
test_sentence = "Tulane University is great."
result = annotate(sentence=test_sentence)

print("Result:", result)
print("Sentiment:", result.sentiment)

# 💡 Hint: The result has a .sentiment attribute
# We'll use this in a loop to annotate multiple sentences


## Part 1c – Step 3: Summarize

Finally, let's test the summarize step that takes all the sentiment labels and creates a summary.


In [ ]:
# @title Test the Summarize Step
# Create the summarize predictor
summarize = dspy.Predict('sentiments: list[str] -> summary: str')

# Test it with a list of sentiment labels
test_sentiments = ["positive", "negative", "positive"]
result = summarize(sentiments=test_sentiments)

print("Result:", result)
print("Summary:", result.summary)

# 💡 Hint: The result has a .summary attribute
# This will be our final output!


## 📋 What Your Pipeline Should Do (Example)

Here's what a working pipeline should produce:

**Input text:**
```
Tulane University announced a major renovation project.
Some alumni are excited, calling it a smart investment.
A few students are frustrated by the temporary noise.
```

**Expected output:**
- **Extracted sentences:** ["Tulane University announced...", "Some alumni are excited...", "A few students are frustrated..."]
- **Sentiment labels:** ["neutral", "positive", "negative"]
- **Final summary:** "The overall sentiment is mixed, with positive reactions from alumni but some negative feedback from students."

Now let's build the complete pipeline that does all three steps!


In [8]:
# @title Part 1d – Build the Complete Pipeline (Your turn!)

def build_sentiment_pipeline_demo(verbose: bool = False):
    """
    Build a sentiment analysis pipeline with three steps:
    1. Extract sentences about Tulane University
    2. Annotate each sentence with sentiment
    3. Summarize overall sentiment
    """
    # Create the three predictors (we tested these above!)
    extract = dspy.Predict('text -> sentences_about_tulane: list[str]')
    annotate = dspy.Predict('sentence -> sentiment: str')
    summarize = dspy.Predict('sentiments: list[str] -> summary: str')

    def pipeline(text: str):
        # Step 1: Extract sentences
        # TODO: Call extract() with the text, then get the sentences from the result
        # Hint: What attribute did the result have in Part 1a?
        sentences = []  # Replace this!
        if verbose:
            print('🟢 Extracted Sentences:', sentences)

        # Step 2: Annotate each sentence
        # TODO: Loop through sentences and call annotate() for each one
        # Hint: You'll need to collect the sentiment labels in a list
        labels = []  # Replace this!
        if verbose:
            print('🟣 Sentiment Labels:', labels)

        # Step 3: Summarize
        # TODO: Call summarize() with the labels, then get the summary
        # Hint: What attribute did the result have in Part 1c?
        summary = 'TODO'  # Replace this!
        if verbose:
            print('🧩 Final Summary:', summary)

        return summary

    return pipeline


<details>
<summary>💡 Stuck? Click here for more hints</summary>

**For Step 1 (Extract):**
- Look back at Part 1a - how did you call `extract()`?
- What did you access from the result to get the sentences?

**For Step 2 (Annotate):**
- You'll need a loop to process each sentence
- For each sentence, call `annotate()` and collect the sentiment
- Look back at Part 1b to see how to get the sentiment from the result

**For Step 3 (Summarize):**
- Call `summarize()` with your list of sentiment labels
- Look back at Part 1c to see how to get the summary from the result

**Remember:** Each `dspy.Predict()` call returns an object with attributes matching the output field names in the signature!

</details>


# Part 2 – Implement safe_run() to Handle Failures

Sometimes things go wrong! The `safe_run()` function wraps a predictor call in error handling so your pipeline doesn't crash.


In [11]:
# @title Implement safe_run()

def safe_run(predictor, **kwargs):
    """
    Wrap a predictor call in try/except to handle errors gracefully.

    TODO: Complete this function:
    1. Try to call the predictor with the kwargs (done)
    2. Check if the result is empty (None or empty string/list)
    3. If there's an error or empty result, print a warning and return None
    4. Otherwise, return the result

    Hint: Use a try/except block. In the try block, call the predictor and check if the result is empty.
    """
    try:
        result = predictor(**kwargs)
        # TODO: Your code here
        pass

    except Exception as e:
        # TODO: Your code here
        pass

# Part 3 – Test Your Pipeline

Let's test your pipeline on a sample article. If you see empty lists or "TODO", go back and complete the pipeline function!


In [ ]:
# @title Run Your Pipeline
article = '''
Tulane University announced a major renovation project.
Some alumni are excited, calling it a smart investment.
A few students are frustrated by the temporary noise and construction.
Overall, Tulane continues to grow in positive ways.
'''

pipeline = build_sentiment_pipeline_demo(verbose=True)
summary = pipeline(article)
print('\n🧠 Final Summary:', summary)

# 💡 If you see empty lists or "TODO", check your pipeline function above!


# Part 4 – Compare to a Single-Prompt Approach

Now let's create a single-prompt version that does everything in one go. This will help you see the difference between modular pipelines and monolithic prompts.


In [ ]:
# @title Write Your Single-Prompt Version
from openai import OpenAI
import os

client = OpenAI()

article = '''
Tulane University announced a major renovation project.
Some alumni are excited, calling it a smart investment.
A few students are frustrated by the temporary noise and construction.
Overall, Tulane continues to grow in positive ways.
'''

# TODO: Write a single prompt that does all three steps at once
# Your prompt should ask the model to:
# 1. Extract sentences about Tulane
# 2. Label each sentence's sentiment
# 3. Summarize the overall sentiment
prompt = '''TODO: your single-prompt text goes here'''

# TODO: Send this prompt to the model using the OpenAI client
# Hint: Look back at Lab 1 if you need a reminder of how to call the API
single_summary = 'TODO'  # Replace this!

print('🧱 Single-Prompt Summary:', single_summary)


## 🐛 Debugging Exercise (Optional but Recommended)

**Common Error #1: "AttributeError: '...' object has no attribute 'sentences_about_tulane'"**

This happens when you forget to call the predictor! Make sure you're doing:
```python
result = extract(text=text)  # Call the function first!
sentences = result.sentences_about_tulane  # Then access the attribute
```

**Common Error #2: Empty sentences list**

If `sentences` is empty, try:
- Check what `extract(text=text)` actually returns
- Use `dspy.inspect_history()` to see what prompt was sent
- Make sure your text mentions "Tulane University" (the extractor looks for this)

**Common Error #3: Pipeline returns "TODO"**

Make sure you:
- Called `summarize(sentiments=labels)`
- Got the result: `result = summarize(sentiments=labels)`
- Extracted the summary: `summary = result.summary`

**Try this:** Add `print()` statements in your pipeline to see what each step produces!


# Part 5 – Add Intermediate Logging

Logging helps you see what's happening at each step. This is especially useful for debugging!


In [ ]:
# @title Add Logging to Your Pipeline
import pandas as pd

# TODO: Modify your pipeline to record intermediate results
#
# Hint: In your pipeline function, after Step 2 (annotate), you have:
#   - sentences (list of strings)
#   - labels (list of sentiment strings)
#
# Create a list of dictionaries where each dictionary has 'sentence' and 'sentiment' keys

interactions = []  # TODO: Fill this with your pipeline's intermediate results

# If you want to test the DataFrame display, try this example:
# interactions = [
#     {'sentence': 'Tulane is great.', 'sentiment': 'positive'},
#     {'sentence': 'Parking is limited.', 'sentiment': 'negative'}
# ]

df = pd.DataFrame(interactions)
print("Logged interactions:")
display(df)

# 💡 Tip: You can modify your pipeline function to return both the summary AND the interactions


## 🧾 Results & Reflection

Answer these questions based on what you observed:

**Results:**
1. How did your pipeline perform? Did it extract sentences correctly? Label sentiment accurately?
2. What happened when errors occurred? Did `safe_run()` help catch them?
3. How did the single-prompt and pipeline versions differ?
   - Which was easier to debug when something went wrong?
   - Which showed you more intermediate steps?

**Conclusion:**
1. Was your hypothesis supported? Why or why not?
   - Did the modular pipeline make debugging easier? Give a specific example.
2. What difference did your single-prompt wording make?
   - Did changing the prompt change the output? How?
3. **Key takeaway:** In one sentence, what's the main advantage of a modular pipeline over a single prompt?


# Part 6 — Run Your Sentiment Pipeline in Gradio

Now let's create an interactive web app! This lets you test your pipeline with different inputs.


In [ ]:
# @title Launch Your Gradio App
import gradio as gr

# Make sure your pipeline is working first!
pipeline = build_sentiment_pipeline_demo(verbose=True)

def run_pipeline(article):
    """This function will be called when the user submits text in Gradio."""
    return pipeline(article)

# Create the Gradio interface
demo = gr.Interface(
    fn=run_pipeline,
    inputs=gr.Textbox(
        label='Paste text mentioning Tulane University',
        lines=4,
        placeholder='Paste or type your article here...'
    ),
    outputs='text',
    title='🧠 DSPY Sentiment Pipeline Explorer',
    description='Analyze sentiment about Tulane using your DSPY pipeline. Try different texts!',
)

# Launch the app
demo.launch(share=True)

# 💡 Tip: Try testing with different texts:
# - Text that mentions Tulane
# - Text that doesn't mention Tulane
# - Very short text
# - Very long text


---

## 🧠 AI Usage Log

> Use this section to document any generative AI assistance (e.g., ChatGPT, Claude, Copilot) you used while completing this lab or assignment.  
> Be specific — transparency and reflection matter more than the amount of AI use.


| Tool Used | Purpose | Prompt / Context | Verification & Edits |
|------------|----------|------------------|----------------------|
| (e.g., ChatGPT (GPT-5)) | (e.g., debugging, code explanation, idea generation) | (e.g., "Why does my cosine similarity return NaN?") | (e.g., ran tests on sample input, compared with lecture code) |
| (Add rows as needed) | | | |

**Summary (2–3 sentences):**  
Briefly describe what you learned or how AI helped you think through the problem.  
Example: *AI helped me notice an off-by-one error in my indexing. I double-checked by printing intermediate results and confirmed the fix.*

---



In [ ]:
# @title ✅ Run Checks for Lab 3

print("🔍 Running Enhanced Lab 3 Checks...\n")

# Check 1: Pipeline existence
try:
    p = build_sentiment_pipeline_demo(verbose=False)
    print("✅ Function build_sentiment_pipeline_demo() exists.")
except Exception as e:
    print("❌ Could not find build_sentiment_pipeline_demo():", e)

# Check 2: Pipeline run behavior
try:
    result = p("Tulane has great professors but limited parking.")
    assert isinstance(result, str), "Pipeline did not return a string."
    if len(result.strip()) > 0:
        print("✅ Pipeline returned a valid (non-empty) summary.")
    else:
        print("⚠️ Pipeline output seems empty. Did you implement summarize()?")
except Exception as e:
    print("❌ Pipeline execution failed:", e)

# Check 3: Extraction test
try:
    result_tulane = p("Tulane is great.")
    result_random = p("Other schools are fine.")
    if result_tulane != result_random:
        print("✅ Extraction step seems to react to 'Tulane' mentions.")
    else:
        print("⚠️ Extraction may not be filtering sentences correctly.")
except Exception as e:
    print("❌ Could not test extraction step:", e)

# Check 4: Summarization
try:
    if "TODO" in result:
        print("⚠️ Your pipeline summary still contains placeholder text ('TODO').")
    else:
        print("✅ Summarization step implemented.")
except Exception as e:
    print("❌ Summarization test failed:", e)

# Check 5: Single-prompt customization
try:
    if "TODO" in prompt or "your single-prompt" in prompt:
        print("⚠️ You have not replaced the placeholder text in your single-prompt prompt variable.")
    else:
        print("✅ Single-prompt text customized.")
except NameError:
    print("⚠️ Could not find 'prompt' variable. Did you define it in Part 4?")

print("\n🏁 Checks complete — review any ⚠️ or ❌ messages above.")
